# Best-of-N

**Paper**: [WebGPT: Browser-assisted question-answering with human feedback](https://arxiv.org/abs/2112.09332)

**Authors**: Reiichiro Nakano, Jacob Hilton, Suchir Balaji, Jeff Wu, Long Ouyang, Christina Kim, Christopher Hesse, Shantanu Jain, Vineet Kosaraju, William Saunders, Xu Jiang, Karl Cobbe, Tyna Eloundou, Gretchen Krueger, Kevin Button, Matthew Knight, Benjamin Chess, John Schulman

Best-of-N sampling is the standard inference-time alignment baseline. It samples several full-length continuations from the base model and returns the single highest-scoring one under a supplied sequence scorer. Pairing the scorer with a majority-vote scorer recovers self-consistency; pairing it with a metric scorer gives metric-guided reranking.

Best-of-N is a decoding driver built on the generic search driver, mapping onto a single search iteration (`num_candidates=n`, `keep_k=1`, `max_iterations=1`, `propose_mode="sample"`) whose one segment spans the whole `max_new_tokens` budget. Each sampled continuation is a full rollout, so any composed logits processor (for example RAD) steers every sample. Parameters for the scorer travel to it at inference time via `runtime_kwargs={"reward_params": {...}}`.

## Method parameters

| parameter | type | description |
| --------- | ---- | ----------- |
| `n` | `int` | Number of full-length continuations to sample and rank |
| `scorer` | `Callable` | A sequence scorer `(prompt, continuations, params) -> list[float]`; the highest-scoring sample is returned |

## Setup

If running this from a Google Colab notebook, uncomment the clone cell below. It is not necessary when running from a virtual environment where the package is already installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub:

In [2]:
# !pip install -q python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: reranking by keyword coverage

The scorer is any callable `(prompt, continuations, params) -> list[float]`, where `params` is whatever was passed as `reward_params` at generation time. We define a scorer that counts how many required keywords a continuation covers, then ask for a single sentence that works in all of them.

In [3]:

from transformers import AutoTokenizer, set_seed

from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.output_control.best_of_n.control import BestOfN

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

/dccstor/principled_ai/users/erikmiehling/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The scorer rewards one point per covered keyword. With ten required words and a 56-token budget, a single sample always drops a few of them, which gives reranking something to do.

In [4]:
def keyword_coverage(prompt, continuations, params):
    terms = [t.lower() for t in params.get("key_terms", [])]
    return [float(sum(term in c.lower() for term in terms)) for c in continuations]


KEY_TERMS = ["cat", "couch", "sun", "tea", "nap", "book", "rain", "socks", "lamp", "blanket"]

### Baseline: a single sample (`n=1`)

With one candidate, taking the argmax over one score is a no-op, so `n=1` is plain sampling. We fix the seed so the runs below are comparable, and print the scorer's verdict alongside the output.

In [5]:
pipeline_n1 = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=1, scorer=keyword_coverage)],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline_n1.steer()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
prompt = (
    "Write one sentence about a lazy afternoon at home that mentions all of these words: "
    "cat, couch, sun, tea, nap, book, rain, socks, lamp, blanket."
)
chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt").to(pipeline_n1.model.device)

set_seed(42)
output = pipeline_n1.generate(
    input_ids=inputs["input_ids"],
    runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
    max_new_tokens=56,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
text_n1 = tokenizer.decode(output[0], skip_special_tokens=True)
print(text_n1)
print("\nkeyword score:", keyword_coverage(prompt, [text_n1], {"key_terms": KEY_TERMS})[0])


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/338 [00:00<04:21,  1.29it/s]


Loading weights:   1%|▏         | 5/338 [00:00<00:46,  7.21it/s]


Loading weights:   5%|▌         | 17/338 [00:01<00:12, 26.44it/s]


Loading weights:   8%|▊         | 26/338 [00:01<00:11, 26.34it/s]

...


Loading weights:  95%|█████████▍| 321/338 [00:07<00:00, 55.76it/s]


Loading weights:  97%|█████████▋| 327/338 [00:07<00:00, 46.38it/s]


Loading weights:  98%|█████████▊| 332/338 [00:07<00:00, 46.59it/s]


Loading weights: 100%|██████████| 338/338 [00:07<00:00, 45.18it/s]

On a rainy afternoon, my fluffy cat curled up on the comfy couch to enjoy a warm cup of tea while I napped under a cozy blanket, surrounded by books and illuminated by the soft glow of the lamp.

keyword score: 8.0


A single sample is at the mercy of the sampling path it happens to take; the score records how many of the ten keywords it covered.

### Best of 8

Same seed, same prompt, but the driver now proposes eight full continuations, scores each with `keyword_coverage`, and returns the argmax.

In [6]:
pipeline_n8 = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=8, scorer=keyword_coverage)],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline_n8.steer()

set_seed(42)
output = pipeline_n8.generate(
    input_ids=inputs["input_ids"].to(pipeline_n8.model.device),
    runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
    max_new_tokens=56,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
text_n8 = tokenizer.decode(output[0], skip_special_tokens=True)
print(text_n8)
print("\nkeyword score:", keyword_coverage(prompt, [text_n8], {"key_terms": KEY_TERMS})[0])


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/338 [00:00<03:34,  1.57it/s]


Loading weights:   1%|▏         | 5/338 [00:00<00:39,  8.52it/s]


Loading weights:   5%|▌         | 17/338 [00:00<00:10, 30.14it/s]


Loading weights:   8%|▊         | 28/338 [00:00<00:06, 45.83it/s]

...


Loading weights:  90%|█████████ | 305/338 [00:04<00:00, 91.30it/s]


Loading weights:  94%|█████████▍| 317/338 [00:04<00:00, 92.75it/s]


Loading weights:  97%|█████████▋| 328/338 [00:04<00:00, 91.68it/s]


Loading weights: 100%|██████████| 338/338 [00:04<00:00, 68.12it/s]

On a lazy afternoon, I lounged on the cozy couch with my favorite cat, sipping hot tea under a soft blanket while reading a book in the rain, surrounded by my beloved socks and illuminated by a warm lamp.

keyword score: 8.0


With eight candidates to choose from, the returned continuation covers more of the required keywords than the single sample did. Nothing about the model changed; the improvement comes entirely from selection.

### What the driver does internally

One search iteration is nothing more than propose, score, keep. The cell below reproduces it directly against the same model: sample eight continuations with `num_return_sequences=8`, score them with the same scorer, and take the argmax. `BestOfN` automates exactly this loop, and generalizes it, since the pipeline's composed logits processors and stopping criteria apply to every rollout.

In [7]:
set_seed(42)
rollouts = pipeline_n8.model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=56,
    do_sample=True,
    num_return_sequences=8,
    pad_token_id=tokenizer.eos_token_id,
)
continuations = tokenizer.batch_decode(rollouts[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
scores = keyword_coverage(prompt, continuations, {"key_terms": KEY_TERMS})

for score, continuation in sorted(zip(scores, continuations), reverse=True):
    print(f"[{score:.0f}] {continuation}")

[8] On a rainy day, I lazily lounged on my cozy couch with a steaming cup of tea, surrounded by a pile of books, while my fluffy cat napped nearby, enjoying the warm sunlight streaming through the window and the soft glow of my bedside lamp casting a comforting
[8] On a rainy afternoon, I lazily snuggled on the cozy sofa with my favorite book while sipping a steaming cup of tea, napping peacefully under the soft glow of my bedside lamp as I read about cats lounging on fluffy blankets in sunny gardens.
[8] On a lazy afternoon, the fluffy cat napped on the cozy couch while sipping tea under the warm lamp, surrounded by books and blankets, as rain pattered softly outside.
[8] On a lazy afternoon, I lounged on the cozy couch with my favorite cat, sipping hot tea under a soft blanket while reading a book in the rain, surrounded by my beloved socks and illuminated by a warm lamp.
[7] On this lazy afternoon, I curled up with my favorite book on the cozy couch, sipping hot tea while napping un

The spread across the eight samples is the whole story of best-of-N. Every rollout misses at least a couple of the ten words, the best cover the most, and the driver simply keeps the top row of this list.

### Scaling `n`

Each candidate is a full rollout, so best-of-N costs `n` times the decode compute of a single generation. The sweep below reads out what that compute buys on this task.

In [8]:
for n in [1, 4, 16]:
    sweep_pipeline = SteeringPipeline(
        model_name_or_path=MODEL_NAME,
        controls=[BestOfN(n=n, scorer=keyword_coverage)],
        device_map="auto",
        hf_model_kwargs={"dtype": "auto"},
    )
    sweep_pipeline.steer()
    set_seed(42)
    output = sweep_pipeline.generate(
        input_ids=inputs["input_ids"].to(sweep_pipeline.model.device),
        runtime_kwargs={"reward_params": {"key_terms": KEY_TERMS}},
        max_new_tokens=56,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    score = keyword_coverage(prompt, [text], {"key_terms": KEY_TERMS})[0]
    print(f"n={n:>2}  winner score: {score:.0f}/10")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights:   8%|▊         | 28/338 [00:00<00:01, 267.21it/s]


Loading weights:  18%|█▊        | 61/338 [00:00<00:00, 302.70it/s]


Loading weights:  27%|██▋       | 92/338 [00:00<00:00, 283.19it/s]


Loading weights:  37%|███▋      | 124/338 [00:00<00:00, 285.94it/s]

...


Loading weights:  75%|███████▌  | 254/338 [00:01<00:00, 286.34it/s]


Loading weights:  84%|████████▎ | 283/338 [00:01<00:00, 282.84it/s]


Loading weights:  93%|█████████▎| 315/338 [00:01<00:00, 285.36it/s]


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 252.92it/s]

n=16  winner score: 9/10


The winner's score improves with `n` and then saturates; past a point, a larger pool mostly resamples the same near-best coverage instead of finding sentences that work in every word. This score-versus-compute curve is the practical dial of the method.

## Example: self-consistency with `MajorityVoteScorer`

Swapping the scorer changes the method. `MajorityVoteScorer` scores each continuation by how many of the others share its extracted answer, so best-of-N with this scorer returns a continuation carrying the plurality answer over `n` sampled reasoning paths. This is self-consistency (Wang et al., 2022), obtained purely as a scorer choice. The scorer takes an `answer_extractor`; here we anchor on the response's final `Answer:` line, with a last-number fallback.

In [9]:
import re

from aisteer360.algorithms.output_control.common.scorers import MajorityVoteScorer


def extract_answer(text: str) -> str:
    match = re.search(r"Answer:\s*\$?(-?\d+(?:\.\d+)?)", text)
    if match:
        return str(float(match.group(1)))
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return str(float(numbers[-1])) if numbers else ""


majority_pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[BestOfN(n=8, scorer=MajorityVoteScorer(answer_extractor=extract_answer))],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
majority_pipeline.steer()

math_prompt = (
    "If it takes 5 machines 5 minutes to make 5 widgets, how many minutes would it take 100 machines "
    'to make 100 widgets? Work through it step by step, then end your response with "Answer: <number>".'
)
math_chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": math_prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
math_inputs = tokenizer(math_chat, return_tensors="pt").to(majority_pipeline.model.device)

set_seed(42)
output = majority_pipeline.generate(
    input_ids=math_inputs["input_ids"],
    max_new_tokens=300,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,
)
majority_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(majority_text)
print("\nextracted answer:", extract_answer(majority_text))


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/338 [00:00<01:05,  5.16it/s]


Loading weights:   8%|▊         | 28/338 [00:00<00:02, 115.37it/s]


Loading weights:  18%|█▊        | 62/338 [00:00<00:01, 192.09it/s]


Loading weights:  26%|██▋       | 89/338 [00:00<00:01, 215.62it/s]

...


Loading weights:  76%|███████▌  | 256/338 [00:01<00:00, 289.44it/s]


Loading weights:  86%|████████▌ | 291/338 [00:01<00:00, 298.63it/s]


Loading weights:  95%|█████████▌| 322/338 [00:01<00:00, 295.34it/s]


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 254.39it/s]

Firstly, let's analyze the given information:

- It takes 5 machines 5 minutes to make 5 widgets.

From this, we can deduce that:
- All 5 machines working together make 5 widgets in 5 minutes.
- Therefore, each machine makes 1 widget in 5 minutes when all 5 machines are working together.

Now, if there are 100 machines instead of 5 and they need to make 100 widgets, we follow these steps:

1. Since one machine makes 1 widget in 5 minutes, 100 machines will also make 1 widget in 5 minutes (because they are working simultaneously).

2. To make 100 widgets, since 100 widgets require 100 machines making 1 widget each in 5 minutes, it will still take them 5 minutes because they are working at full capacity.

Therefore, the answer is: 5.

extracted answer: 5.0


### Comparison: a single greedy answer

The self-consistency claim is that the plurality over sampled reasoning paths beats the single path greedy decoding commits to. For the comparison we decode the same prompt greedily, without the driver.

In [10]:
greedy_ids = majority_pipeline.model.generate(
    input_ids=math_inputs["input_ids"],
    attention_mask=math_inputs["attention_mask"],
    max_new_tokens=300,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
greedy_text = tokenizer.decode(greedy_ids[0][math_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(greedy_text)
print("\nextracted answer:", extract_answer(greedy_text))

To solve this problem, let's break it down step by step:

1. **Understand the given information:**
   - 5 machines can make 5 widgets in 5 minutes.

2. **Determine the rate of work for one machine:**
   - Since 5 machines can make 5 widgets in 5 minutes, each machine makes \( \frac{5}{5} = 1 \) widget in 5 minutes.
   - Therefore, each machine works at a rate of making 1 widget per minute.

3. **Calculate the total number of widgets made by 100 machines in 5 minutes:**
   - If one machine can make 1 widget in 1 minute, then 100 machines will make \( 100 \times 1 = 100 \) widgets in 5 minutes.

4. **Conclusion:**
   - It would still take 5 minutes for 100 machines to make 100 widgets because their combined rate is still 1 widget per minute.

Therefore, the answer is:
Answer: 5

extracted answer: 5.0


The correct answer is 5 minutes (each machine makes one widget in 5 minutes, so 100 machines make 100 widgets in the same 5 minutes). Note that both greedy and best-of-n answer this particular question correctly. The value of self-consistency is with respect to robustness, i.e., individual samples do occasionally fail and as problems get harder, the benefit of multiple sampled paths increases (Wang et al.).

### Takeaway

Best-of-N is the first thing to try when you have a clear score since it needs no training, composes with everything, and costs a transparent `n` full decodes per output. The scorer is the method, as this notebook shows twice with the same driver (keyword reranking, then self-consistency via `MajorityVoteScorer`; the shipped scorers live in `aisteer360.algorithms.output_control.common.scorers`).

Because every candidate is a full rollout, a step-level control steers all `n` samples. Running RAD under `BestOfN` reranks already-detoxified candidates ([rad.ipynb](rad.ipynb)). For iterative segment-level search with the same scorer contract, see DeAL ([deal.ipynb](deal.ipynb)).